## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

Wed Aug  5 07:36:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone repo + cài thư viện (idempotent)

In [ ]:
%cd /content
![ -d Locality-iN-Locality ] || git clone https://github.com/Omid-Nejati/Locality-iN-Locality.git
%cd /content/Locality-iN-Locality
!pip -q install torchattacks timm einops torchsummary matplotlib

/content
Cloning into 'Locality-iN-Locality'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 45 (delta 4), reused 2 (delta 2), pack-reused 40 (from 1)
Receiving objects: 100% (45/45), 45.99 KiB | 9.20 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/Locality-iN-Locality
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.7/178.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 15.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of th

In [ ]:
import os, sys, time, json, csv, shutil
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import WeightedRandomSampler

import torchattacks
from torchattacks import PGD, FGSM

print("PyTorch", torch.__version__, "| Torchvision", torchvision.__version__,
      "| Torchattacks", torchattacks.__version__)

PyTorch 2.11.0+cu128 | Torchvision 0.26.0+cu128 | Torchattacks 3.5.1


## 2b. Khóa seed (tái lập kết quả) — không đổi recipe

In [ ]:
import random

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # giữ tốc độ; muốn tất định tuyệt đối thì đổi thành False + deterministic=True (chậm hơn)
    torch.backends.cudnn.benchmark = True

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(SEED)
g = torch.Generator(); g.manual_seed(SEED)
print('Seed fixed:', SEED)

Seed fixed: 42


## 3. Tải + giải nén GTSRB (chỉ tải nếu chưa có, unzip không hỏi)

In [ ]:
%%bash
mkdir -p data
BASE=https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370

[ -f data/GTSRB_Final_Training_Images.zip ] || curl -L $BASE/GTSRB_Final_Training_Images.zip -o data/GTSRB_Final_Training_Images.zip
[ -f data/GTSRB_Final_Test_Images.zip ]     || curl -L $BASE/GTSRB_Final_Test_Images.zip     -o data/GTSRB_Final_Test_Images.zip
[ -f data/GTSRB_Final_Test_GT.zip ]         || curl -L $BASE/GTSRB_Final_Test_GT.zip         -o data/GTSRB_Final_Test_GT.zip

unzip -q -o data/GTSRB_Final_Training_Images.zip -d data/
unzip -q -o data/GTSRB_Final_Test_Images.zip -d data/
unzip -q -o data/GTSRB_Final_Test_GT.zip -d data/
echo "Done download & unzip."


Done download & unzip.


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  263M  100  263M    0     0  11.7M      0  0:00:22  0:00:22 --:--:-- 14.4M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 84.8M  100 84.8M    0     0  10.6M      0  0:00:07  0:00:07 --:--:-- 12.6M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 99620  100 99620    0     0  90366      0  0:00:01  0:00:01 --:--:-- 90399


## 4. Sắp xếp test set theo class (43 lớp) — idempotent

In [ ]:
import os, shutil

data_dir = './data/GTSRB'
images_dir = os.path.join(data_dir, 'Final_Test/Images')
test_dir = os.path.join(data_dir, 'test')
os.makedirs(test_dir, exist_ok=True)

# Chỉ sắp xếp lại nếu chưa đủ 43 lớp (chạy lại không copy thừa)
if len([d for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d))]) < 43:
    with open('./data/GT-final_test.csv') as f:
        lines = f.readlines()
    for text in lines[1:]:
        cls = int(text.split(';')[-1])
        name = text.split(';')[0]
        cls_dir = os.path.join(test_dir, f'{cls:04d}')
        os.makedirs(cls_dir, exist_ok=True)
        dst = os.path.join(cls_dir, name)
        if not os.path.exists(dst):
            shutil.copy(os.path.join(images_dir, name), cls_dir)

print('Số class test:', len([d for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d))]))

Số class test: 43


## 5. Transforms + DataLoader  — GIỮ NGUYÊN từ 28-07-improved (99.64%)

Chuẩn hoá ImageNet, WeightedRandomSampler (1/√count), augment nhẹ hợp biển báo (không lật ngang).

In [ ]:
batch_size = 64  # giữ nguyên như bản 99.64%

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomApply([transforms.RandomRotation(12)], p=0.5),
    transforms.RandomApply([transforms.RandomAffine(degrees=0, translate=(0.08, 0.08),
                                                    scale=(0.9, 1.1), shear=8)], p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12)),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.ImageFolder(root='./data/GTSRB/Final_Training/Images', transform=train_transform)
testset  = torchvision.datasets.ImageFolder(root='./data/GTSRB/test', transform=test_transform)

targets = np.array(trainset.targets)
class_sample_count = np.array([len(np.where(targets == t)[0]) for t in np.unique(targets)])
weight = 1. / np.sqrt(class_sample_count)
samples_weight = torch.from_numpy(weight[targets]).double()
sampler = WeightedRandomSampler(samples_weight, len(samples_weight), generator=g)

train_loader = torch.utils.data.DataLoader(dataset=trainset, batch_size=batch_size,
                                           sampler=sampler, num_workers=2, pin_memory=True,
                                           worker_init_fn=seed_worker, generator=g)
test_loader  = torch.utils.data.DataLoader(dataset=testset, batch_size=batch_size,
                                           shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(trainset)} ảnh, Test: {len(testset)} ảnh, {len(trainset.classes)} lớp')

Train: 39209 ảnh, Test: 12630 ảnh, 43 lớp


## 6. Model LNL-Ti (43 lớp) — GIỮ NGUYÊN từ bản 99.64%

In [ ]:
from LNL import LNL_Ti as small

model = small(pretrained=False)
model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)
model = model.cuda()
print(f'Số tham số: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

/usr/local/lib/python3.12/dist-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/Locality-iN-Locality/models/deit.py:78: UserWarning: Overwriting deit_tiny_patch16_224 in registry with models.deit.deit_tiny_patch16_224. This is because the name being registered

Số tham số: 6.08M


## 7. Mount Drive + tạo thư mục lưu kết quả cho lần chạy này

In [ ]:
from google.colab import drive
import glob
drive.mount('/content/drive')

# RESUME=True  -> dùng lại folder run gần nhất (giữ checkpoint đã train, KHÔNG train lại)
# RESUME=False -> tạo run mới để train từ đầu
RESUME = True

runs_root = '/content/drive/MyDrive/LNL_runs'
os.makedirs(runs_root, exist_ok=True)
existing = sorted(glob.glob(f'{runs_root}/run_*'))

if RESUME and existing:
    RUN_DIR = existing[-1]                      # folder run mới nhất đã có
    print('RESUME — dùng lại:', RUN_DIR)
else:
    RUN_DIR = f"{runs_root}/run_{time.strftime('%Y%m%d_%H%M')}"
    os.makedirs(RUN_DIR, exist_ok=True)
    print('Tạo run mới:', RUN_DIR)

ckpt_path = f'{RUN_DIR}/lnl_ti_gtsrb_best.pth'
print('Có checkpoint sẵn:', os.path.exists(ckpt_path))


Mounted at /content/drive
Tạo run mới: /content/drive/MyDrive/LNL_runs/run_20260805_0738
Có checkpoint sẵn: False


## 8. Cấu hình training — GIỮ NGUYÊN từ 28-07-improved (99.64%)

15 epochs, AdamW (lr=5e-4, wd=0.05), OneCycleLR (max_lr=1e-3, warmup 30%), label smoothing 0.05.

In [ ]:
num_epochs = 15

loss = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3,
    steps_per_epoch=len(train_loader), epochs=num_epochs,
    pct_start=0.3, div_factor=10, final_div_factor=100,
)

# Lưu cấu hình làm bằng chứng
config = dict(model='LNL_Ti', epochs=num_epochs, batch_size=batch_size,
              optimizer='AdamW', lr=5e-4, weight_decay=0.05,
              scheduler='OneCycleLR', max_lr=1e-3, pct_start=0.3,
              label_smoothing=0.05, grad_clip=1.0, normalize='imagenet',
              sampler='WeightedRandomSampler(1/sqrt(count))')
json.dump(config, open(f'{RUN_DIR}/config.json', 'w'), indent=2)
print(config)

{'model': 'LNL_Ti', 'epochs': 15, 'batch_size': 64, 'optimizer': 'AdamW', 'lr': 0.0005, 'weight_decay': 0.05, 'scheduler': 'OneCycleLR', 'max_lr': 0.001, 'pct_start': 0.3, 'label_smoothing': 0.05, 'grad_clip': 1.0, 'normalize': 'imagenet', 'sampler': 'WeightedRandomSampler(1/sqrt(count))'}


## 9. Train — CHỈ TRAIN, lưu checkpoint mỗi epoch

Vòng train không đánh giá test (nhanh hơn). Mỗi epoch lưu 1 checkpoint đầy đủ
(model + optimizer + scheduler) vào `{RUN_DIR}/checkpoints/` để có thể resume
và để phần Test đánh giá lại từng epoch.

### Hàm evaluate (dùng ở phần Test)

In [ ]:
@torch.no_grad()
def evaluate(net, loader):
    net.eval()
    correct = total = 0
    for images, labels in loader:
        images = images.cuda()
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels.cuda()).sum().item()
    return 100.0 * correct / total

In [ ]:
# ============ TRAIN ONLY — lưu checkpoint mỗi epoch ============
# TRAIN=True  -> train (tự resume nếu đã có ckpt_last.pth trong RUN_DIR)
# TRAIN=False -> bỏ qua train, dùng checkpoint đã lưu
TRAIN = True

import time, os, glob

CKPT_DIR = f'{RUN_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
LAST_CKPT = f'{CKPT_DIR}/ckpt_last.pth'
TRAIN_LOG = f'{RUN_DIR}/train_log.csv'

def save_ckpt(path, epoch, train_loss, train_acc):
    torch.save({
        'epoch': epoch,                                  # số epoch ĐÃ hoàn thành
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'train_loss': train_loss,
        'train_acc': train_acc,
        'config': config,
    }, path)

start_epoch = 0
if TRAIN and os.path.exists(LAST_CKPT):
    ck = torch.load(LAST_CKPT, map_location='cuda', weights_only=False)
    model.load_state_dict(ck['model_state'])
    optimizer.load_state_dict(ck['optimizer_state'])
    scheduler.load_state_dict(ck['scheduler_state'])
    start_epoch = ck['epoch']
    print(f'RESUME từ epoch {start_epoch} (đã train {start_epoch}/{num_epochs})')

# header cho log (chỉ ghi 1 lần)
if not os.path.exists(TRAIN_LOG):
    with open(TRAIN_LOG, 'w') as f:
        f.write('epoch,train_loss,train_acc,lr,sec\n')

if not TRAIN:
    print('Bỏ qua training — dùng checkpoint tại', CKPT_DIR)
elif start_epoch >= num_epochs:
    print('Đã train đủ', num_epochs, 'epoch — không train thêm.')
else:
    for epoch in range(start_epoch, num_epochs):
        model.train()
        t0 = time.time()
        total_loss, run_correct, run_total = 0.0, 0, 0

        for i, (batch_images, batch_labels) in enumerate(train_loader):
            X = batch_images.cuda()
            Y = batch_labels.cuda()

            pre = model(X)
            cost = loss(pre, Y)

            optimizer.zero_grad()
            cost.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            total_loss += cost.item()
            _, pred = torch.max(pre.data, 1)
            run_total += Y.size(0)
            run_correct += (pred == Y).sum().item()

        train_loss = total_loss / len(train_loader)
        train_acc  = 100.0 * run_correct / run_total
        lr_now     = scheduler.get_last_lr()[0]
        secs       = time.time() - t0

        # ---- lưu checkpoint CỦA TỪNG EPOCH + bản last (để resume) ----
        save_ckpt(f'{CKPT_DIR}/ckpt_epoch_{epoch+1:02d}.pth', epoch + 1, train_loss, train_acc)
        save_ckpt(LAST_CKPT, epoch + 1, train_loss, train_acc)

        with open(TRAIN_LOG, 'a') as f:
            f.write(f'{epoch+1},{train_loss:.4f},{train_acc:.2f},{lr_now:.3e},{secs:.0f}\n')

        print(f'Epoch [{epoch+1}/{num_epochs}]  Loss: {train_loss:.4f}  '
              f'Train: {train_acc:.2f}%  lr: {lr_now:.2e}  ({secs:.0f}s)  -> đã lưu ckpt_epoch_{epoch+1:02d}.pth')

    print('\nTrain xong. Checkpoint đã lưu:', len(glob.glob(f'{CKPT_DIR}/ckpt_epoch_*.pth')), 'epoch')

## 10. Test — kiểm tra kết quả TỪNG EPOCH

Duyệt qua tất cả checkpoint đã lưu, đánh giá test accuracy cho từng epoch,
ghi ra `eval_log.csv`, rồi chọn epoch tốt nhất và lưu thành `lnl_ti_gtsrb_best.pth`.
Đã đánh giá rồi thì bỏ qua (chạy lại không tốn thời gian).

In [ ]:
# ============ TEST từng epoch ============
import glob, os, csv, time

CKPT_DIR  = f'{RUN_DIR}/checkpoints'
EVAL_LOG  = f'{RUN_DIR}/eval_log.csv'
BEST_PATH = f'{RUN_DIR}/lnl_ti_gtsrb_best.pth'

# đọc lại các epoch đã đánh giá trước đó (tránh chạy lại)
done = {}
if os.path.exists(EVAL_LOG):
    with open(EVAL_LOG) as f:
        r = csv.DictReader(f)
        for row in r:
            done[int(row['epoch'])] = float(row['test_acc'])
else:
    with open(EVAL_LOG, 'w') as f:
        f.write('epoch,train_loss,train_acc,test_acc\n')

ckpts = sorted(glob.glob(f'{CKPT_DIR}/ckpt_epoch_*.pth'))
print(f'Tìm thấy {len(ckpts)} checkpoint\n')

for p in ckpts:
    ep = int(os.path.basename(p).split('_')[-1].split('.')[0])
    if ep in done:
        print(f'Epoch {ep:02d}  Test: {done[ep]:.2f}%  (đã đánh giá trước đó)')
        continue

    ck = torch.load(p, map_location='cuda', weights_only=False)
    model.load_state_dict(ck['model_state'])

    t0 = time.time()
    acc = evaluate(model, test_loader)
    done[ep] = acc

    with open(EVAL_LOG, 'a') as f:
        f.write(f"{ep},{ck.get('train_loss', float('nan')):.4f},"
                f"{ck.get('train_acc', float('nan')):.2f},{acc:.2f}\n")

    print(f"Epoch {ep:02d}  Loss: {ck.get('train_loss', float('nan')):.4f}  "
          f"Train: {ck.get('train_acc', float('nan')):.2f}%  Test: {acc:.2f}%  ({time.time()-t0:.0f}s)")

# ---- chọn epoch tốt nhất ----
if done:
    best_epoch = max(done, key=done.get)
    best_acc   = done[best_epoch]
    print(f'\n>>> Epoch tốt nhất: {best_epoch}  —  Test accuracy: {best_acc:.2f}%')

    ck_best = torch.load(f'{CKPT_DIR}/ckpt_epoch_{best_epoch:02d}.pth',
                         map_location='cuda', weights_only=False)
    torch.save(ck_best['model_state'], BEST_PATH)   # weights-only, gọn để dùng lại
    print('Đã lưu weights tốt nhất ->', BEST_PATH)

In [ ]:
# Bảng tổng hợp từng epoch
import pandas as pd
df = pd.read_csv(f'{RUN_DIR}/eval_log.csv').sort_values('epoch')
display(df)

# load lại checkpoint tốt nhất để dùng cho phần robust bên dưới
model.load_state_dict(torch.load(BEST_PATH, map_location='cuda'))
std_acc = float(df['test_acc'].max())
print(f'Standard accuracy (best epoch): {std_acc:.2f} %')

## 11. Robust accuracy — FGSM và PGD (chạy trên checkpoint tốt nhất)

Model chuẩn hoá bằng mean/std = 0.5 nên phải khai báo `set_normalization_used`
đúng giá trị đó để tấn công trong không gian pixel [0,1].

In [ ]:
# Model được Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) ở mục 5
# -> phải khai báo ĐÚNG mean/std đó thì attack mới đúng không gian pixel [0,1]
NORM_MEAN = [0.5, 0.5, 0.5]
NORM_STD  = [0.5, 0.5, 0.5]

def robust_eval(atk, loader):
    model.eval()
    correct = total = 0
    for images, labels in loader:
        adv = atk(images, labels).cuda()
        _, pred = torch.max(model(adv), 1)
        total += labels.size(0)
        correct += (pred == labels.cuda()).sum().item()
    return 100.0 * correct / total

atk = FGSM(model, eps=0.01)
atk.set_normalization_used(mean=NORM_MEAN, std=NORM_STD)
fgsm_acc = robust_eval(atk, test_loader)
print(f'FGSM robust accuracy: {fgsm_acc:.2f} %')

atk = PGD(model, eps=0.01, alpha=2/255, steps=5, random_start=False)
atk.set_normalization_used(mean=NORM_MEAN, std=NORM_STD)
pgd_acc = robust_eval(atk, test_loader)
print(f'PGD robust accuracy: {pgd_acc:.2f} %')

## 12. Biểu đồ + lưu kết quả cuối vào Drive (bằng chứng)

In [ ]:
import pandas as pd

df = pd.read_csv(f'{RUN_DIR}/eval_log.csv').sort_values('epoch')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(df['epoch'], df['train_loss'], marker='o')
ax[0].set_title('Train loss'); ax[0].set_xlabel('Epoch'); ax[0].grid(alpha=0.3)
ax[1].plot(df['epoch'], df['test_acc'], marker='o', label='Test')
if 'train_acc' in df:
    ax[1].plot(df['epoch'], df['train_acc'], marker='s', alpha=0.6, label='Train')
ax[1].set_title('Accuracy (%)'); ax[1].set_xlabel('Epoch'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.savefig(f'{RUN_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

best_epoch = int(df.loc[df['test_acc'].idxmax(), 'epoch'])
results = dict(standard_accuracy=std_acc, fgsm_accuracy=fgsm_acc, pgd_accuracy=pgd_acc,
               best_accuracy=float(df['test_acc'].max()), best_epoch=best_epoch,
               total_epochs=int(df['epoch'].max()),
               per_epoch_test_acc={int(r.epoch): float(r.test_acc) for r in df.itertuples()})
json.dump(results, open(f'{RUN_DIR}/results.json', 'w'), indent=2)
print('Đã lưu toàn bộ kết quả vào:', RUN_DIR)
print(results)